# 🧠 SVM Breast Cancer Classification Notebook

This notebook demonstrates how to apply **Support Vector Machines (SVM)** to classify tumors as Malignant or Benign using the Breast Cancer Wisconsin Diagnostic Dataset.

We will cover:
1. Data Loading & Preprocessing
2. Model Pipelines & Benchmarking
3. Hyperparameter Tuning with GridSearchCV
4. Confusion Matrix Evaluation
5. PCA Decision Boundary Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

### 1. Load Data
We load the `breast-cancer.csv` dataset, drop the `id` column, and encode the `diagnosis` as binary variables (M=1, B=0).

In [ ]:
df = pd.read_csv('breast-cancer.csv')
df.drop(columns=['id'], inplace=True, errors='ignore')
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
df.head()

### 2. Pipeline & Training
We use an `sklearn` Pipeline to scale the data and train the SVM.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42))
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

### 3. Confusion Matrix Visualization

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title('Confusion Matrix - SVM (RBF)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### 4. PCA Decision Boundary
To visualize our decision boundaries, we use PCA to reduce the features to 2 dimensions.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

svm_2d = SVC(kernel='rbf', C=1.0, gamma='scale').fit(X_pca, y)

h = .02
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.coolwarm)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, edgecolors='k', cmap=plt.cm.coolwarm)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('SVM RBF Kernel Decision Boundary (PCA)')
handles, labels = scatter.legend_elements()
plt.legend(handles, ['Benign', 'Malignant'])
plt.show()